# ====================================
#      Shopify E-Commerce Analysis
#       August 2025 - July 2026
# ====================================

# Analysis Scope

This analysis examines Shopify order data in the period Aug 2025 - Jul 2026

Focus Topics:
    - overall sales performance
    - monthly sales and discount trends
    - customer purchasing behavior
    - product family performance
    - geographic sales distribution
    - refunds and return-related patterns

In Shopify, each line-item in an order is tracked in separate rows. 
Line-item-level and order-level data are analyzed separately where appropriate to avoid double counting transactions.

Customer-level analysis uses e-mail as customer identifier.



In [1]:
# 1. Setup and Data Loading

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/shopify_orders.csv")

df.shape

(12039, 77)

In [2]:
# 2. Data Cleaning and Validation

df["Gross line value"] = (df["Lineitem quantity"] * df["Lineitem price"])

# Establish two data grains
line_items = df.copy()
orders = df.loc[df["Subtotal"].notna()].copy()

order_cols = [
    "Name",
    "Created at",
    "Email",
    "Billing Name",
    "Billing City",
    "Billing Province",
    "Billing Country",
    "Currency",
    "Financial Status",
    "Fulfillment Status",
    "Subtotal",
    "Shipping",
    "Taxes",
    "Total",
    "Discount Code",
    "Discount Amount",
    "Refunded Amount",
    "Payment Method"
]

orders_clean = orders[order_cols].copy()

# Date conversion
orders_clean["Created at"] = (
    pd.to_datetime(
        orders_clean["Created at"],
        utc=True
    ).dt.tz_convert("America/Los_Angeles")
)


# Define the final analysis period:
# August 1, 2025 through July 31, 2026
analysis_start = pd.Timestamp("2025-08-01", tz="America/Los_Angeles")
analysis_end = pd.Timestamp("2026-08-01", tz="America/Los_Angeles")

orders_clean = orders_clean[
    (orders_clean["Created at"] >= analysis_start)
    & (orders_clean["Created at"] < analysis_end)
].copy()

# Filter line items to analysis period
line_items = line_items[
    line_items["Name"].isin(orders_clean["Name"])
].copy()

# Make month field
orders_clean["Year Month"] = (
    orders_clean["Created at"]
    .dt.strftime("%Y-%m")
)

# Create order-level variables
units_per_order = (
    line_items
    .groupby("Name")["Lineitem quantity"]
    .sum()
    .reset_index(name="units")
)

orders_clean = orders_clean.merge(
    units_per_order,
    on="Name",
    how="left"
)

# Pre-discount merchandise value
orders_clean["gross_order_item_value"] = (
    orders_clean["Subtotal"]
    + orders_clean["Discount Amount"]
)

# Classify selected major SKUs into more general product families
sku = line_items["Lineitem sku"].fillna("")

line_items["product_style"] = np.select(
    [
        sku.str.startswith("200-"),
        sku.str.startswith("207-"),
        sku.str.startswith("205-")
        | sku.str.startswith("CX205"),
        sku.str.startswith("105-"),
        sku.str.startswith("551-"),
        sku.str.startswith("109-MT-")
    ],
    [
        "Product E",
        "Product F",
        "Product D",
        "Product C",
        "Product B",
        "Product A"
    ],
    default="Other / unclassified"
)

# Order Check Table - verify gross order subtotal agrees with gross line-items value minus order discounts.
order_check = (
    line_items.groupby("Name")
      .agg(
          gross_value=("Gross line value", "sum"),
          discount_amount=("Discount Amount", "first"),
          subtotal=("Subtotal", "first"),
      ).reset_index()
)

order_check["discount_amount"] = (
    order_check["discount_amount"].fillna(0)
)

order_check["expected_subtotal"] = (
    order_check["gross_value"]
    - order_check["discount_amount"]
)

order_check["subtotal_difference"] = (
    order_check["subtotal"]
    - order_check["expected_subtotal"]
)

order_check["subtotal_difference"].abs().max()

# Verify there's one row per order
orders_clean["Name"].nunique(), len(orders_clean)

# Verify date range
orders_clean["Created at"].agg(["min", "max"])

min   2025-08-01 04:32:15-07:00
max   2026-07-31 21:05:12-07:00
Name: Created at, dtype: datetime64[us, America/Los_Angeles]

In [3]:
# 3. Overall Business Performance

total_orders = len(orders_clean)

gross_item_sales = line_items["Gross line value"].sum()
total_discounts = orders_clean["Discount Amount"].sum()
net_item_sales = orders_clean["Subtotal"].sum()

total_shipping = orders_clean["Shipping"].sum()
total_taxes = orders_clean["Taxes"].sum()
total_order_value = orders_clean["Total"].sum()
total_refunds = orders_clean["Refunded Amount"].sum()

average_order_total = orders_clean["Total"].mean()
average_order_subtotal = orders_clean["Subtotal"].mean()

orders_with_refunds = (orders_clean["Refunded Amount"] > 0).sum()
orders_with_discounts = (orders_clean["Discount Amount"] > 0).sum()
refunded_order_pct = (orders_with_refunds / total_orders * 100)
discounted_order_pct = (orders_with_discounts / total_orders * 100)

overall_performance = pd.Series({
        "Total orders": f"{total_orders:,}",
        "Gross item sales": f"${gross_item_sales:,.2f}",
        "Total discounts": f"${total_discounts:,.2f}",
        "Net item sales": f"${net_item_sales:,.2f}",
        "Total shipping": f"${total_shipping:,.2f}",
        "Total taxes": f"${total_taxes:,.2f}",
        "Total order value": f"${total_order_value:,.2f}",
        "Total refunds": f"${total_refunds:,.2f}",
        "Average order total": f"${average_order_total:,.2f}",
        "Average order subtotal": f"${average_order_subtotal:,.2f}",
        "Orders with refunds": f"{orders_with_refunds:,}",
        "Refunded order %": f"{refunded_order_pct:.2f}%",
        "Orders with discounts": f"{orders_with_discounts:,}",
        "Discounted order %": f"{discounted_order_pct:.2f}%"
    })

overall_performance

Total orders                      9,499
Gross item sales    [REDACTED]
Total discounts    [REDACTED]
Net item sales    [REDACTED]
Total shipping    [REDACTED]
Total taxes    [REDACTED]
Total order value    [REDACTED]
Total refunds    [REDACTED]
Average order total    [REDACTED]
Average order subtotal    [REDACTED]
Orders with refunds               1,296
Refunded order %                 13.64%
Orders with discounts             5,259
Discounted order %               55.36%
dtype: str

In [4]:
# 4. Monthly Sales and Discount Trends

monthly_sales = (
    orders_clean
    .groupby("Year Month")
    .agg(
        orders=("Name", "count"),
        net_item_sales=("Subtotal", "sum"),
        total_order_value=("Total", "sum"),
        discounts=("Discount Amount", "sum"),
    )
    .reset_index()
)

monthly_sales["average_order_total"] = (
    monthly_sales["total_order_value"]
    / monthly_sales["orders"]
)

monthly_sales["gross_item_sales"] = (
    monthly_sales["net_item_sales"]
    + monthly_sales["discounts"]
)

monthly_sales["discount_rate"] = (
    monthly_sales["discounts"]
    / monthly_sales["gross_item_sales"]
)

monthly_sales["discount_rate_pct"] = (
    monthly_sales["discount_rate"] * 100
)

# Nov/Dec investigation
nov_dec = (
    orders_clean[
        orders_clean["Year Month"].isin(
            ["2025-11", "2025-12"]
        )
    ]
    .groupby("Year Month")
    .agg(
        orders=("Name", "count"),
        net_item_sales=("Subtotal", "sum"),
        average_units=("units", "mean"),
        average_gross_item_value=(
            "gross_order_item_value", "mean"
        ),
        average_discount=("Discount Amount", "mean"),
        average_subtotal=("Subtotal", "mean")
    )
)

nov_dec_change_pct = (
    (
        nov_dec.loc["2025-12"]
        / nov_dec.loc["2025-11"]
    ) - 1
) * 100

nov_dec_change_display = nov_dec_change_pct.map(
    lambda x: f"{x:.2f}%"
)

nov_dec_change_display

orders                       9.34%
net_item_sales              -0.48%
average_units               -1.01%
average_gross_item_value    -3.98%
average_discount            45.49%
average_subtotal            -8.98%
dtype: str

In [5]:
# 5. Product Performance

# 5a. SKU Performance and Product family stability

selected_families = [
    "Product E",
    "Product F",
    "Product D",
    "Product C",
    "Product B",
    "Product A"
]

sku_performance = (
    line_items
    .dropna(subset=["Lineitem sku"])
    .groupby("Lineitem sku")
    .agg(
        units_sold=("Lineitem quantity", "sum"),
        gross_sales=("Gross line value", "sum")
    )
    .reset_index()
    .sort_values("gross_sales", ascending=False)
)

sku_performance["sales_share_pct"] = (
    sku_performance["gross_sales"]
    / sku_performance["gross_sales"].sum()
    * 100
)

sku_performance["cumulative_sales_pct"] = (
    sku_performance["sales_share_pct"].cumsum()
)

order_context = (orders_clean[["Name", "Created at", "Refunded Amount"]]
    .rename(columns={
        "Created at": "Order date",
        "Refunded Amount": "Order Refunded Amount"
    }))

product_analysis = line_items.merge(
    order_context,
    on="Name",
    how="left",
    validate="many_to_one"
)

product_analysis["Month"] = (
    product_analysis["Order date"]
    .dt.tz_localize(None)
    .dt.to_period("M")
)

product_analysis["Was Refunded"] = (
    product_analysis["Order Refunded Amount"] > 0
)

product_analysis = product_analysis[
    product_analysis["product_style"].isin(selected_families)
    ].copy()

# Monthly units by product family
monthly_family_units = (
    product_analysis
    .groupby(["product_style", "Month"])["Lineitem quantity"]
    .sum()
    .unstack(fill_value=0)
)

# Ensure all 12 analysis months are represented
analysis_months = pd.period_range(
    "2025-08",
    "2026-07",
    freq="M"
)

monthly_family_units = (
    monthly_family_units
    .reindex(columns=analysis_months, fill_value=0)
)

# Coefficient of variation across the 12-month analysis period
family_stability = pd.DataFrame({
    "avg_monthly_units": monthly_family_units.mean(axis=1),
    "monthly_unit_std": monthly_family_units.std(axis=1, ddof=0)
})

family_stability["unit_cv"] = (
    family_stability["monthly_unit_std"]
    / family_stability["avg_monthly_units"]
)

display(sku_performance.head(10))

family_stability.sort_values("unit_cv")

,Lineitem sku,units_sold,gross_sales,sales_share_pct,cumulative_sales_pct
278,207-5W,59,[REDACTED],1.310539,1.310539
273,207-1C,50,[REDACTED],1.110626,2.421166
253,200-2B,57,[REDACTED],0.980156,3.401322
254,200-2K,56,[REDACTED],0.978361,4.379683
260,200-5W,59,[REDACTED],0.953528,5.333211
251,200-1N,55,[REDACTED],0.888501,6.221712
270,205-1N,48,[REDACTED],0.872819,7.094531
123,109-MT-42,46,[REDACTED],0.835999,7.930530
271,205-WC,29,[REDACTED],0.732004,8.662533
252,200-2#8,40,[REDACTED],0.724676,9.387209


,avg_monthly_units,monthly_unit_std,unit_cv
product_style,,,
Product E,38.166667,9.762456,0.255785
Product D,16.250000,4.798003,0.295262
Product B,7.250000,2.772634,0.382432
Product F,12.333333,4.731690,0.383651
Product C,10.916667,4.499228,0.412143
Product A,20.333333,14.243907,0.700520


In [6]:
# 5b. Product family performance and Refunds

family_sales_summary = (
    product_analysis
    .groupby("product_style")
    .agg(
        units=("Lineitem quantity", "sum"),
        gross_sales=("Gross line value", "sum"),
        unique_skus=("Lineitem sku", "nunique")
    )
    .reset_index()
    .sort_values("gross_sales", ascending=False)
)

family_sales_summary["gross_sales_per_unit"] = (
    family_sales_summary["gross_sales"]
    / family_sales_summary["units"]
)

# One row per order-family combination for refund association
family_order_level = (
    product_analysis[
        ["Name", "product_style", "Was Refunded"]
        ]
    .drop_duplicates(
        subset=["Name", "product_style"]
    )
)

family_refund_summary = (
    family_order_level
    .groupby("product_style")
    .agg(
        orders=("Name", "nunique"),
        refunded_orders=("Was Refunded", "sum")
    )
    .reset_index()
)

family_refund_summary["refunded_order_pct"] = (
    family_refund_summary["refunded_orders"]
    / family_refund_summary["orders"]
    * 100
)

product_family_summary = (
    family_sales_summary
    .merge(
        family_refund_summary,
        on="product_style"
    )
    .sort_values("gross_sales",ascending=False)
)

product_family_summary

,product_style,units,gross_sales,unique_skus,gross_sales_per_unit,orders,refunded_orders,refunded_order_pct
0,Product E,458,[REDACTED],16,[REDACTED],433,43,9.930716
1,Product A,244,[REDACTED],20,[REDACTED],211,67,31.753555
2,Product D,195,[REDACTED],10,[REDACTED],184,19,10.326087
3,Product F,148,[REDACTED],8,[REDACTED],140,13,9.285714
4,Product C,131,[REDACTED],7,[REDACTED],117,15,12.820513
5,Product B,87,[REDACTED],3,[REDACTED],81,12,14.814815


In [7]:
# 6. Customer Analysis

# Keep orders that have an email attached
customer_orders = orders_clean[orders_clean["Email"].notna()].copy()

# Summarize purchase behavior for each email
customer_summary = (
    customer_orders
    .groupby("Email")
    .agg(
        orders=("Name", "count"),
        net_item_sales=("Subtotal", "sum"),
        first_order=("Created at", "min"),
        last_order=("Created at", "max")
    )
    .reset_index()
)

# Is customer one-time or repeat?
customer_summary["customer_type"] = np.where(
    customer_summary["orders"] == 1,
    "One-time",
    "Repeat"
)

# Compare size and purchase activity of one-time vs repeat customers
customer_type_summary = (
    customer_summary
    .groupby("customer_type")
    .agg(
        customers=("Email", "count"),
        orders=("orders", "sum"),
        net_item_sales=("net_item_sales", "sum")
    )
    .reset_index()
)

# Calculate each customer type's share of customer/order/merchandise sales
customer_type_summary["customer_share_pct"] = (
    customer_type_summary["customers"]
    / customer_type_summary["customers"].sum()
    * 100
)

customer_type_summary["order_share_pct"] = (
    customer_type_summary["orders"]
    / customer_type_summary["orders"].sum()
    * 100
)

customer_type_summary["sales_share_pct"] = (
    customer_type_summary["net_item_sales"]
    / customer_type_summary["net_item_sales"].sum()
    * 100
)

customer_orders = customer_orders.merge(
    customer_summary[["Email", "customer_type"]],
    on="Email",
    how="left"
)

customer_type_summary["average_sales_per_customer"] = (
    customer_type_summary["net_item_sales"]
    / customer_type_summary["customers"]
)

repeat_order_distribution = (
    customer_summary[
        customer_summary["customer_type"] == "Repeat"
        ]
    .groupby("orders")
    .size()
    .reset_index(name="customers")
    .sort_values("orders")
)

# Calculate % of repeats that fall into each order count group
repeat_order_distribution["customer_share_pct"] = (
    repeat_order_distribution["customers"]
    / repeat_order_distribution["customers"].sum()
    * 100
)

# Compare discount behavior between one-time and repeat customers
customer_orders["used_discount"] = (customer_orders["Discount Amount"] > 0)

customer_orders["discount_status"] = np.where(
    customer_orders["Discount Amount"] > 0,
    "Discounted",
    "No discount"
)

customer_discount_summary = (
    customer_orders
    .groupby("customer_type")
    .agg(
        orders=("Name", "count"),
        discounted_orders=("used_discount", "sum"),
        total_discounts=("Discount Amount", "sum"),
        net_item_sales=("Subtotal", "sum")
    )
    .reset_index()
)

customer_discount_summary["average_discount_per_order"] = (
    customer_discount_summary["total_discounts"]
    / customer_discount_summary["orders"]
)
 
customer_discount_summary["gross_item_sales"] = (
    customer_discount_summary["net_item_sales"]
    + customer_discount_summary["total_discounts"]
)

customer_discount_summary["discount_rate_pct"] = (
    customer_discount_summary["total_discounts"]
    / customer_discount_summary["gross_item_sales"]
    * 100
)

customer_discount_summary["discounted_order_pct"] = (
    customer_discount_summary["discounted_orders"]
    / customer_discount_summary["orders"]
    * 100
)

# Compare discounted and non-discounted orders within each customer type
discount_behavior_comparison = (
    customer_orders
    .groupby(["customer_type", "discount_status"])
    .agg(
        orders=("Name", "count"),
        average_gross_item_value=("gross_order_item_value", "mean"),
        average_discount=("Discount Amount", "mean"),
        average_order_subtotal=("Subtotal", "mean")
    )
    .reset_index()
)

# Focus only on customers who placed at least 2 orders
repeat_customer_summary = customer_summary[
    customer_summary["orders"] >= 2
    ].copy()

repeat_customer_summary["order_segment"] = np.select(
    [
        repeat_customer_summary["orders"] == 2,
        repeat_customer_summary["orders"] == 3,
        repeat_customer_summary["orders"] >= 4
    ],
    [
        "2 orders",
        "3 orders",
        "4+ orders"
    ],
    default="Unknown"
)

repeat_segment_summary = (
    repeat_customer_summary
    .groupby("order_segment")
    .agg(
        customers=("Email", "count"),
        orders=("orders", "sum"),
        net_item_sales=("net_item_sales", "sum")
    )
    .reset_index()
)

repeat_segment_summary["avg_sales_per_customer"] = (
    repeat_segment_summary["net_item_sales"]
    / repeat_segment_summary["customers"]
)

repeat_segment_summary["avg_order_subtotal"] = (
    repeat_segment_summary["net_item_sales"]
    / repeat_segment_summary["orders"]
)
repeat_segment_summary["repeat_customer_share_pct"] = (
    repeat_segment_summary["customers"]
    / repeat_segment_summary["customers"].sum()
    * 100
)

repeat_segment_summary["repeat_sales_share_pct"] = (
    repeat_segment_summary["net_item_sales"]
    / repeat_segment_summary["net_item_sales"].sum()
    * 100
)

repeat_segment_summary

,order_segment,customers,orders,net_item_sales,avg_sales_per_customer,avg_order_subtotal,repeat_customer_share_pct,repeat_sales_share_pct
0,2 orders,940,1880,[REDACTED],[REDACTED],[REDACTED],67.238913,46.541758
1,3 orders,246,738,[REDACTED],[REDACTED],[REDACTED],17.596567,18.439865
2,4+ orders,212,1218,[REDACTED],[REDACTED],[REDACTED],15.164521,35.018376


In [8]:
# 7. Geographic Analysis

# Summarize order activity by billing country
country_performance = (
    orders_clean
    .groupby("Billing Country")
    .agg(
        orders=("Name", "count"),
        net_item_sales=("Subtotal", "sum")
    )
    .reset_index()
    .sort_values("net_item_sales", ascending=False)
)

country_performance["order_share_pct"] = (
    country_performance["orders"]
    / country_performance["orders"].sum()
    * 100
)

country_performance["sales_share_pct"] = (
    country_performance["net_item_sales"]
    / country_performance["net_item_sales"].sum()
    * 100
)

display(country_performance.head(10))


# Classify orders as US or International
orders_clean["market"] = np.where(
    orders_clean["Billing Country"] == "US",
    "United States",
    "International"
)

market_performance = (
    orders_clean
    .groupby("market")
    .agg(
        orders=("Name", "count"),
        net_item_sales=("Subtotal", "sum"),
        average_order_subtotal=("Subtotal", "mean"),
        median_order_subtotal=("Subtotal", "median"),
        order_subtotal_std=("Subtotal", "std")                           
    )
    .reset_index()
)

market_performance["order_share_pct"] = (
    market_performance["orders"]
    / market_performance["orders"].sum()
    * 100
)

market_performance["sales_share_pct"] = (
    market_performance["net_item_sales"]
    / market_performance["net_item_sales"].sum()
    * 100
)

display(market_performance.head(10))


# Keep only US orders with a known billing province/state
us_orders = orders_clean[
    (orders_clean["Billing Country"] == "US")
    & (orders_clean["Billing Province"].notna())
    ].copy()

# Summarize order activity by US state
state_performance = (
    us_orders
    .groupby("Billing Province")
    .agg(
        orders=("Name", "count"),
        net_item_sales=("Subtotal", "sum"),
        average_order_subtotal=("Subtotal", "mean")
    )
    .reset_index()
    .sort_values("net_item_sales", ascending=False)
)

state_performance["order_share_pct"] = (
    state_performance["orders"]
    / state_performance["orders"].sum()
    * 100
)

state_performance["sales_share_pct"] = (
    state_performance["net_item_sales"]
    / state_performance["net_item_sales"].sum()
    * 100
)

state_performance.head(10)

,Billing Country,orders,net_item_sales,order_share_pct,sales_share_pct
23,US,9334,[REDACTED],98.574295,98.440931
2,CA,62,[REDACTED],0.654768,0.628166
0,AU,8,[REDACTED],0.084486,0.131512
3,CH,10,[REDACTED],0.105608,0.118527
8,GB,10,[REDACTED],0.105608,0.117671
14,KW,2,[REDACTED],0.021122,0.062440
10,IL,5,[REDACTED],0.052804,0.061974
11,IS,2,[REDACTED],0.021122,0.049745
12,JP,4,[REDACTED],0.042243,0.048143
5,DE,5,[REDACTED],0.052804,0.044526


,market,orders,net_item_sales,average_order_subtotal,median_order_subtotal,order_subtotal_std,order_share_pct,sales_share_pct
0,International,165,[REDACTED],[REDACTED],[REDACTED],[REDACTED],1.737025,1.859586
1,United States,9334,[REDACTED],[REDACTED],[REDACTED],[REDACTED],98.262975,98.140414


,Billing Province,orders,net_item_sales,average_order_subtotal,order_share_pct,sales_share_pct
6,CA,1318,[REDACTED],[REDACTED],14.182718,13.073587
46,TX,985,[REDACTED],[REDACTED],10.599376,11.425863
11,FL,554,[REDACTED],[REDACTED],5.961476,5.679844
36,NY,486,[REDACTED],[REDACTED],5.229743,5.072898
16,IL,336,[REDACTED],[REDACTED],3.615625,3.723987
12,GA,309,[REDACTED],[REDACTED],3.325083,3.657547
7,CO,289,[REDACTED],[REDACTED],3.109868,3.226164
29,NC,287,[REDACTED],[REDACTED],3.088346,3.186611
48,VA,289,[REDACTED],[REDACTED],3.109868,2.963496
50,WA,276,[REDACTED],[REDACTED],2.969977,2.809686


In [9]:
# 8. Refund and Return Analysis

orders_clean["order_date"] = orders_clean["Created at"].dt.date

# Flag orders with refunds
orders_clean["was_refunded"] = (
    orders_clean["Refunded Amount"] > 0
)

orders_clean["refund_status"] = np.where(
    orders_clean["was_refunded"],
    "Refunded",
    "No refund"
)

# Define equal 77-day periods around the May 16 return-policy change
before_start = pd.Timestamp("2026-02-28").date()
policy_change = pd.Timestamp("2026-05-16").date()
after_end = pd.Timestamp("2026-07-31").date()

refund_policy_analysis = orders_clean[
    (
        (orders_clean["order_date"] >= before_start)
        & (orders_clean["order_date"] < policy_change)
    )
    |
    (
        (orders_clean["order_date"] >= policy_change)
        & (orders_clean["order_date"] <= after_end)
    )
    ].copy()

refund_policy_analysis["policy_period"] = np.where(
    refund_policy_analysis["order_date"] < policy_change,
    "Before policy change",
    "After policy change"
)

# Compare refund activity before and after the change
refund_policy_summary = (
    refund_policy_analysis
    .groupby("policy_period")
    .agg(
        orders=("Name", "count"),
        refunded_orders=("was_refunded", "sum"),
        total_refunded=("Refunded Amount", "sum"),
        net_item_sales=("Subtotal", "sum")
    )
    .reset_index()
)

refund_policy_summary["refunded_order_pct"] = (
    refund_policy_summary["refunded_orders"]
    / refund_policy_summary["orders"]
    * 100
)

refund_policy_summary["refund_to_sales_pct"] = (
    refund_policy_summary["total_refunded"]
    / refund_policy_summary["net_item_sales"]
    * 100
)

# Extract the before/after rows for easier comparison
before = refund_policy_summary[
    refund_policy_summary["policy_period"] == "Before policy change"
    ].iloc[0]

after = refund_policy_summary[
    refund_policy_summary["policy_period"] == "After policy change"
    ].iloc[0]

# Measure relative changes after the policy update
refund_rate_change_pct = (
    (after["refunded_order_pct"] - before["refunded_order_pct"])
    / before["refunded_order_pct"]
    * 100
)

refund_to_sales_change_pct = (
    (after["refund_to_sales_pct"] - before["refund_to_sales_pct"])
    / before["refund_to_sales_pct"]
    * 100
)

# Create a weekly period based on each order's creation date
refund_weekly = orders_clean.copy()

refund_weekly["order_week"] = (
    refund_weekly["Created at"]
    .dt.tz_localize(None)
    .dt.to_period("W")
    .astype(str)
)

# Calculate weekly refund activity
weekly_refund_summary = (
    refund_weekly
    .groupby("order_week")
    .agg(
        orders=("Name", "count"),
        refunded_orders=("was_refunded", "sum"),
        total_refunded=("Refunded Amount", "sum"),
        net_item_sales=("Subtotal", "sum")
    )
    .reset_index()
)

weekly_refund_summary["refunded_order_pct"] = (
    weekly_refund_summary["refunded_orders"]
    / weekly_refund_summary["orders"]
    * 100
)

weekly_refund_summary["refund_to_sales_pct"] = (
    weekly_refund_summary["total_refunded"]
    / weekly_refund_summary["net_item_sales"]
    * 100
)

# Compare the characteristics of refunded and non-refunded orders
refund_order_characteristics = (
    orders_clean
    .groupby("refund_status")
    .agg(
        orders=("Name", "count"),
        average_units=("units", "mean"),
        median_units=("units", "median"),
        average_gross_item_value=("gross_order_item_value", "mean"),
        average_discount=("Discount Amount", "mean"),
        average_order_subtotal=("Subtotal", "mean"),
        median_order_subtotal=("Subtotal", "median"),
        average_refunded_amount=("Refunded Amount", "mean")
    )
    .reset_index()
)

refunded_orders = orders_clean[
    orders_clean["was_refunded"]
    & (orders_clean["Total"] > 0)
    ].copy()

refunded_orders["refund_share_of_total"] = (
    refunded_orders["Refunded Amount"]
    / refunded_orders["Total"]
)

refund_share_summary = (
    refunded_orders["refund_share_of_total"]
    .describe(
        percentiles=[0.25, 0.5, 0.75]
    )
)

# Flag 90%+ refunded orders
refunded_orders["refund_extent"] = np.where(
    refunded_orders["refund_share_of_total"] >= 0.90,
    "90%+ refunded",
    "Less than 90% refunded"
)

refund_extent_summary = (
    refunded_orders["refund_extent"]
    .value_counts()
    .reset_index()
)

refund_extent_summary.columns = ["refund_extent", "orders"]

refund_extent_summary["order_share_pct"] = (
    refund_extent_summary["orders"]
    / refund_extent_summary["orders"].sum()
    * 100
)

policy_change_summary = pd.Series({
    "Refunded order rate change": f"{refund_rate_change_pct:.2f}%",
    "Refund-to-sales change": f"{refund_to_sales_change_pct:.2f}%"
})

display(refund_policy_summary)
display(policy_change_summary)
display(refund_order_characteristics)
display(refund_extent_summary)

,policy_period,orders,refunded_orders,total_refunded,net_item_sales,refunded_order_pct,refund_to_sales_pct
0,After policy change,1821,195,[REDACTED],[REDACTED],10.708402,8.625823
1,Before policy change,1870,244,[REDACTED],[REDACTED],13.048128,11.666816


Refunded order rate change    -17.93%
Refund-to-sales change        -26.07%
dtype: str

,refund_status,orders,average_units,median_units,average_gross_item_value,average_discount,average_order_subtotal,median_order_subtotal,average_refunded_amount
0,No refund,8203,1.266366,1.0,[REDACTED],[REDACTED],[REDACTED],[REDACTED],[REDACTED]
1,Refunded,1296,1.462963,1.0,[REDACTED],[REDACTED],[REDACTED],[REDACTED],[REDACTED]


,refund_extent,orders,order_share_pct
0,90%+ refunded,722,55.795981
1,Less than 90% refunded,572,44.204019


# 9. Key Findings

### Sales and Order Volume
December 2025 had the highest volume of orders in the analysis period, +9.3% from November. Despite the increase in orders, net merchandise sales stayed fairly flat (-0.5%). The difference was driven by a lower average order subtotal (-9.0%), lower pre-discount cart value (-4.0%) and higher average discounts (+45.5%).

### Customer Value

The business is driven by a core group of repeat customers. Despite repeat customers representing only 19.8% of customers, they generated 45.3% of customer sales. Repeat customers also spent more per transaction, +22% higher than the average one-time customer's average order subtotal. Within this segment, customers that placed 4+ orders represented about 15.2% of repeat customers but drove more than a third (35.0%) of repeat customer sales.

Business implication: Repeat customers contribute disproportionately to sales, suggesting that customer retention and conversion to a second purchase are important areas to monitor.

### Product Performance
The Product E led the selected major product families in unit volume, with 458 units sold and about [REDACTED] in gross sales. Month to month, it also had the lowest variation in unit volume among the selected families.

The Product A had the highest refund association among the selected families - about 31.8% of orders containing that jacket received some refund, compared with an overall refunded order rate of about 13.6%.

Business implication: The approximate 2.33x refund association for the Product A warrants a deeper investigation into its cause. Customer service feedback suggests the texture of the jacket's material is a common reason for returns. 

### Refund and Return Policy
Order refunds went down following the return-policy change that took place May 16, 2026. Equal 77-day periods were compared before and after the policy change, in which the order refund rate fell from about 13.0% to 10.7%, a 17.9% relative decline. Refund dollars relative to sales declined from 11.7% to 8.6%, a 26.1% relative change.

However, order refunds were already moving downward prior to the policy change, so the policy may have contributed to the decline but does not establish causation.

Business implication: The decline is a positive sign, though the existing downward trend means the new refund policy's effect should be monitored longer before attributing the decline to the policy change.

### Refund Characteristics
Refunded orders tended to contain more items and higher merchandise values than non-refunded orders. Among observed refunded orders, more than half had at least 90% of the order's value refunded. For this analysis, these were treated as near-full refunds, since restocking and shipping fees can reduce the recorded refund amount.

### Geographic Context
The business was overwhelmingly U.S. based during the analysis period (about 98.1% of sales share). Geographic differences existed across states, but were less prominent than the customer, product, sales, and refund findings above.

In [10]:
# 10. Power BI Report Exports

# 10a. Refund Analysis Export

# Cleaned order level dataset
refund_powerbi = orders_clean[
    [
        "Name",
        "Created at",
        "Subtotal",
        "Refunded Amount",
    ]
    ].copy()

refund_powerbi["Order Date"] = (
    refund_powerbi["Created at"]
    .dt.tz_localize(None)
    .dt.normalize()
)

refund_powerbi["Was Refunded"] = (
    refund_powerbi["Refunded Amount"] > 0
)

policy_change_date = pd.Timestamp("2026-05-16")

refund_powerbi["Policy Period"] = np.where(
    refund_powerbi["Order Date"] < policy_change_date,
    "Before policy change",
    "After policy change"
)

# Keep only final Power BI fields
refund_powerbi = refund_powerbi[
    [
        "Name",
        "Order Date",
        "Subtotal",
        "Refunded Amount",
        "Was Refunded",
        "Policy Period"
    ]
    ]

# Final preview
refund_powerbi.head()

# Export for Power BI
refund_powerbi.to_csv("refund_powerbi.csv", index=False)

In [11]:
# 10b. Customer Analysis Export

# Create anonymous customer identifier
customer_id_map = {
    email: customer_id
    for customer_id, email in enumerate(
        customer_orders["Email"].dropna().unique(),
        start=1
    )
}

customer_powerbi = customer_orders[
    [
        "Name",
        "Email",
        "customer_type",
        "Subtotal",
        "Discount Amount",
        "Refunded Amount"
    ]
    ].copy()

customer_powerbi["Customer ID"] = (
    customer_powerbi["Email"]
    .map(customer_id_map)
)

# Remove emails before export
customer_powerbi = customer_powerbi[
    [
        "Name",
        "Customer ID",
        "customer_type",
        "Subtotal",
        "Discount Amount",
        "Refunded Amount"
    ]
    ].copy()

# Friendlier Power BI column names
customer_powerbi = customer_powerbi.rename(
    columns={
        "Name": "Order ID",
        "customer_type": "Customer Type"
    }
)

# Preview final export table
customer_powerbi.head()

# Export for Power BI
customer_powerbi.to_csv("customer_powerbi.csv", index=False)

In [12]:
# 10c. Sales & Discounts Export

# Cleaned table
monthly_powerbi = orders_clean[
    [
        "Name",
        "Created at",
        "Subtotal",
        "Discount Amount",
        "gross_order_item_value"
    ]
    ].copy()

monthly_powerbi["Order Date"] = (
    monthly_powerbi["Created at"]
    .dt.tz_localize(None)
    .dt.normalize()
)

# Rename/filter fields for Power BI
monthly_powerbi = monthly_powerbi.rename(
    columns={
        "Name": "Order ID",
        "gross_order_item_value": "Gross Order Item Value"
    }
)

monthly_powerbi = monthly_powerbi[
    [
        "Order ID",
        "Order Date",
        "Subtotal",
        "Discount Amount",
        "Gross Order Item Value"
    ]
    ].copy()

monthly_powerbi.head()

monthly_powerbi.to_csv("monthly_powerbi.csv", index=False)

In [13]:
# 10d. Product Analysis Export

# Add order-level context to line-item data
product_powerbi = line_items[
    [
        "Name",
        "Lineitem sku",
        "Lineitem quantity",
        "Gross line value",
        "product_style"
    ]
    ].copy()

order_context = orders_clean[
    [
        "Name",
        "Created at",
        "Refunded Amount"
    ]
    ].copy()

product_powerbi = product_powerbi.merge(
    order_context,
    on="Name",
    how="left",
    validate="many_to_one"
)

# Create Power BI-friendly fields
product_powerbi["Order Date"] = (
    product_powerbi["Created at"]
    .dt.tz_localize(None)
    .dt.normalize()
)

product_powerbi["Was Refunded"] = (
    product_powerbi["Refunded Amount"] > 0
)

product_powerbi = product_powerbi.rename(
    columns={
        "Name": "Order ID",
        "Lineitem sku": "SKU",
        "Lineitem quantity": "Units",
        "Gross line value": "Gross Sales",
        "product_style": "Product Family"
    }
)

product_powerbi = product_powerbi[
    [
        "Order ID",
        "Order Date",
        "SKU",
        "Product Family",
        "Units",
        "Gross Sales",
        "Was Refunded"
    ]
    ]

product_powerbi.head()

product_powerbi.to_csv("product_powerbi.csv", index=False)